# Week 45, Day 1: The diffusion model as a measurement tool

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Simulating choices and response times

Complete ddm(): on each 1 ms step, add v dt plus noise of standard deviation sqrt(dt) to every undecided trial; mark trials that reach a (choice 1) or 0 (choice 0) and record their response time, including the non-decision time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2008)

def ddm(v, a, ter, n=2000, dt=0.001, z=None):
    x = np.full(n, a / 2 if z is None else z)
    done = np.zeros(n, bool); ch = np.zeros(n, int); rt = np.zeros(n); k = 0
    while not done.all() and k < 10000:
        idx = ~done
        ...
        k += 1
        up, lo = idx & (x >= a), idx & (x <= 0)
        ch[up] = 1; rt[up | lo] = k * dt + ter; done |= up | lo
    return ch, rt

ch, rt = ddm(v=1.0, a=1.2, ter=0.3)
skew = ((rt - rt.mean()) ** 3).mean() / rt.std() ** 3
print(f"accuracy {ch.mean():.3f}; mean RT correct {rt[ch == 1].mean():.3f} s, errors {rt[ch == 0].mean():.3f} s; skewness {skew:.2f}")
plt.hist(rt[ch == 1], 60, alpha=0.6, label="correct"); plt.hist(rt[ch == 0], 60, alpha=0.6, label="error")
plt.xlabel("response time (s)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "response times are right-skewed (skewness above 1) and accuracy is between 0.7 and 0.9"
    assert skew > 1 and 0.7 < ch.mean() < 0.9
def test_floor():
    "no response is faster than the non-decision time"
    assert rt.min() >= 0.3

test_shape()
test_floor()
print("All checks passed")

### Exercise: EZ-diffusion

Write ez(pc, vrt, mrt) from the formulas above and apply it to the simulated data.

In [ ]:
def ez(pc, vrt, mrt):
    L = np.log(pc / (1 - pc))
    v = ...
    a = ...
    mdt = ...
    return v, a, mrt - mdt

v_hat, a_hat, t_hat = ez(ch.mean(), rt[ch == 1].var(), rt[ch == 1].mean())
print(f"EZ estimates: v = {v_hat:.3f} (true 1.0), a = {a_hat:.3f} (true 1.2), Ter = {t_hat:.3f} s (true 0.3)")

In [ ]:
import numpy as np
def test_recover():
    "EZ recovers v within 0.15, a within 0.15 and Ter within 0.03 of the truth"
    v_hat, a_hat, t_hat = ez(ch.mean(), rt[ch == 1].var(), rt[ch == 1].mean())
    assert abs(v_hat - 1.0) < 0.15 and abs(a_hat - 1.2) < 0.15 and abs(t_hat - 0.3) < 0.03

test_recover()
print("All checks passed")

### Exercise: Which parameter moved?

Simulate three conditions: accurate (v 1.0, a 1.2), speed (v 1.0, a 0.8) and hard (v 0.6, a 1.2), all with Ter 0.3 and 4,000 trials. Fit each with ez and print accuracy, mean RT and the three estimates.

In [ ]:
conds = {"accurate": (1.0, 1.2), "speed": (1.0, 0.8), "hard": (0.6, 1.2)}
fits = {}
for name, (v, a) in conds.items():
    c, t = ...
    fits[name] = ez(c.mean(), t[c == 1].var(), t[c == 1].mean())
    print(f"{name:8s} accuracy {c.mean():.3f}, mean RT {t.mean():.3f} s -> v {fits[name][0]:.2f}, a {fits[name][1]:.2f}, Ter {fits[name][2]:.3f}")

In [ ]:
import numpy as np
def test_speed():
    "under a speed instruction, the fitted bound falls by more than 0.25 while drift stays within 0.15"
    assert fits["accurate"][1] - fits["speed"][1] > 0.25 and abs(fits["accurate"][0] - fits["speed"][0]) < 0.15
def test_hard():
    "for a weaker stimulus, the fitted drift falls by more than 0.3 while the bound stays within 0.15"
    assert fits["accurate"][0] - fits["hard"][0] > 0.3 and abs(fits["accurate"][1] - fits["hard"][1]) < 0.15

test_speed()
test_hard()
print("All checks passed")